# NAFP+NMFP Recipe v3 — 3 seeds × 30 epochs × BSZ=320 × NT-Xent

**Pre-registered:** `data/results/nafp/recipe_v3_30ep/PROTOCOL.md`

**Single Colab session, 3 seeds sequentially. Total wall time ~4.5h.**

**Before running:**
1. Runtime → Change runtime type → GPU (L4 preferred, T4 fallback)
2. Upload via left sidebar:
   - `kaggle.json` (Kaggle API token)
   - `nafp_patched_recipe_v3.tar.gz` (from your Mac at `/Users/prita/Desktop/Audio Fingerprinting/afp_bench/`)
3. Run all cells. Don't close the tab.

**Output:** `/content/recipe_v3_3seeds_output.tar.gz` at the end. Download via left sidebar.

## 1. GPU sanity

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
assert out.returncode == 0, 'No GPU. Runtime > Change runtime type > GPU'
print('GPU:', out.stdout.strip())

## 2. Locate sidebar-uploaded files

In [ ]:
import os, shutil, glob
kj = sorted(glob.glob('/content/kaggle*.json'))
tb = sorted(glob.glob('/content/*recipe_v3*.tar.gz'))
assert kj, 'No kaggle.json in /content/'
assert tb, 'No tarball in /content/. Upload nafp_patched_recipe_v3.tar.gz'
print(f'kaggle: {kj[0]} ({os.path.getsize(kj[0])} bytes)')
print(f'tarball: {tb[0]} ({os.path.getsize(tb[0])/1024:.1f} KB)')
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(kj[0], '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
if tb[0] != '/content/nafp_patched_recipe_v3.tar.gz':
    shutil.copy(tb[0], '/content/nafp_patched_recipe_v3.tar.gz')
print('Ready.')

## 3. Install deps (~3 min)

In [ ]:
%pip install -q 'tensorflow==2.19.0' 'tf-keras==2.19.0' 'kapre==0.3.7' 'numpy<2.2' kaggle pyyaml librosa
import tensorflow; print('tf:', tensorflow.__version__)
import kapre; print('kapre:', kapre.__version__)
import tf_keras; print('tf_keras: OK')

## 4. Download FMA-medium dataset (~10 min, 9.84 GB)

In [ ]:
import subprocess, os, glob
if not glob.glob('/content/data/**/music', recursive=True):
    os.makedirs('/content/data', exist_ok=True)
    out = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'mimbres/neural-audio-fingerprint',
         '-p', '/content/data', '--unzip'],
        capture_output=True, text=True, timeout=1800
    )
    print('STDOUT tail:', out.stdout[-1500:])
    print('STDERR tail:', out.stderr[-1500:])
    assert out.returncode == 0, 'Kaggle download failed'
else:
    print('Dataset already downloaded')
candidates = glob.glob('/content/data/**/music', recursive=True)
DATA_ROOT = os.path.dirname(candidates[0])
print(f'DATA_ROOT: {DATA_ROOT}')
n_train = len(glob.glob(f'{DATA_ROOT}/music/train-10k-30s/**/*.wav', recursive=True))
assert n_train >= 9000, f'Only {n_train} train wavs found'
print(f'train_wavs: {n_train}')

## 5. Extract patched repo + verify v3 patches

In [ ]:
import tarfile, os, shutil
REPO_ROOT = '/content/nafp_upstream'
if os.path.exists(REPO_ROOT): shutil.rmtree(REPO_ROOT)
with tarfile.open('/content/nafp_patched_recipe_v3.tar.gz', 'r:gz') as tar:
    tar.extractall('/content/')
os.rename('/content/upstream', REPO_ROOT)
with open(f'{REPO_ROOT}/config/recipe_v3.yaml') as f: cfg_txt = f.read()
for must_have in ['F_MIN : 160.', "TR_SEG_MODE : 'random_oneshot'", 'MAX_EPOCH : 30', 'TR_BATCH_SZ : 320', 'TR_N_ANCHOR : 160']:
    assert must_have in cfg_txt, f'PATCH MISSING: {must_have!r}'
print('All 5 patches verified in recipe_v3.yaml')

## 6. Patch config DIR paths for Colab

In [ ]:
import yaml
cfg_path = f'{REPO_ROOT}/config/recipe_v3.yaml'
with open(cfg_path) as f: cfg = yaml.safe_load(f)
cfg['DIR']['SOURCE_ROOT_DIR'] = f'{DATA_ROOT}/music/'
cfg['DIR']['BG_ROOT_DIR']     = f'{DATA_ROOT}/aug/bg/'
cfg['DIR']['IR_ROOT_DIR']     = f'{DATA_ROOT}/aug/ir/'
cfg['DIR']['SPEECH_ROOT_DIR'] = f'{DATA_ROOT}/aug/speech/common_voice_8k/en/'
cfg['DIR']['OUTPUT_ROOT_DIR'] = '/content/logs/emb/'
cfg['DIR']['LOG_ROOT_DIR']    = '/content/logs/'
cfg.setdefault('DATA_SEL', {})['REDUCE_ITEMS_P'] = 0
cfg.setdefault('TRAIN', {})['MINI_TEST_IN_TRAIN'] = False
with open(cfg_path, 'w') as f: yaml.dump(cfg, f, sort_keys=False)
for k in ['MAX_EPOCH']: print(f"  TRAIN.{k}: {cfg['TRAIN'][k]}")
for k in ['F_MIN']: print(f"  MODEL.{k}: {cfg['MODEL'][k]}")
for k in ['TR_SEG_MODE']: print(f"  DATA_SEL.{k}: {cfg['DATA_SEL'][k]}")
for k in ['TR_BATCH_SZ', 'TR_N_ANCHOR']: print(f"  BSZ.{k}: {cfg['BSZ'][k]}")

## 7. GPU compute sanity

In [ ]:
import os, time
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
assert gpus, 'NO GPU'
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print(f'set_memory_growth: {e}')
with tf.device('/GPU:0'):
    a = tf.random.normal((2048, 2048)); b = tf.random.normal((2048, 2048))
    _ = tf.matmul(a, b).numpy()
    t0 = time.time(); _ = tf.matmul(a, b).numpy()
    ms = (time.time()-t0)*1000
print(f'GPU matmul: {ms:.1f} ms')
assert ms < 200, f'Too slow ({ms}ms) — TF on CPU'

## 8. CosineDecay alias patch for TF 2.19

In [ ]:
from pathlib import Path
tr = Path(REPO_ROOT)/'model'/'trainer.py'
src = tr.read_text()
if 'tf.keras.experimental.CosineDecay' in src:
    tr.write_text(src.replace('tf.keras.experimental.CosineDecay', 'tf.keras.optimizers.schedules.CosineDecay'))
    print('Patched trainer.py')
else:
    print('Already patched')

## 9. OOM smoke test at BSZ=320 (2 steps only, ~30 sec)

If this fails with OOM, drop TR_BATCH_SZ to 240 (or 160) in cell 6 and re-run.

In [ ]:
import subprocess, os
env = {**os.environ, 'TF_USE_LEGACY_KERAS': '1', 'PYTHONUNBUFFERED': '1', 'NAFP_TRAIN_EPOCHS': '1', 'PYTHONHASHSEED': '42'}
result = subprocess.run(
    ['python', '-u', 'run.py', 'train', 'smoke_test', '-c', 'recipe_v3', '--max_epoch=1'],
    cwd=REPO_ROOT, env=env, capture_output=True, text=True, timeout=180
)
print('STDOUT tail:', result.stdout[-1500:])
if 'OOM' in result.stdout or 'OOM' in result.stderr:
    print('!!! OOM at BSZ=320. Reduce TR_BATCH_SZ in cell 6 and re-run from cell 6. !!!')
    raise SystemExit(1)
print('OOM smoke OK — proceeding to 3-seed training')
# Clean smoke checkpoint
import shutil
shutil.rmtree('/content/logs/checkpoint/smoke_test', ignore_errors=True)

## 10. Train 3 seeds sequentially (~4.5h total)

Seed 42 → seed 137 → seed 2026. Each ~90 min. Watch loss decrease.

In [ ]:
import subprocess, time, os, shutil, glob
SEEDS = [42, 137, 2026]
results = {}
for seed in SEEDS:
    exp_name = f'recipe_v3_seed{seed}'
    print(f'\n{"="*60}\n  SEED {seed} starting at {time.strftime("%H:%M:%S")}\n{"="*60}', flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        ['python', '-u', 'run.py', 'train', exp_name, '-c', 'recipe_v3', f'--max_epoch=30'],
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True,
        env={**os.environ, 'TF_USE_LEGACY_KERAS': '1', 'PYTHONUNBUFFERED': '1', 'PYTHONHASHSEED': str(seed)}
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    elapsed = (time.time() - t0) / 60
    print(f'\nSEED {seed} finished in {elapsed:.1f} min, exit={proc.returncode}')
    assert proc.returncode == 0, f'SEED {seed} failed'
    # Verify ckpt-30 exists
    ckpt_idx = sorted([int(p.split('ckpt-')[1].split('.')[0])
                       for p in glob.glob(f'/content/logs/checkpoint/{exp_name}/ckpt-*.index')])
    assert 30 in ckpt_idx or max(ckpt_idx) == 30, f'ckpt-30 not found for {exp_name}'
    results[seed] = elapsed

print(f'\nAll 3 seeds done. Per-seed minutes: {results}')

## 11. Bundle all 3 checkpoints for download

Download `recipe_v3_3seeds_output.tar.gz` via the left Files sidebar.

In [ ]:
import tarfile, json, os, shutil, time, glob
BUNDLE = '/content/recipe_v3_3seeds_output'
if os.path.exists(BUNDLE): shutil.rmtree(BUNDLE)
os.makedirs(BUNDLE)
shutil.copy(f'{REPO_ROOT}/config/recipe_v3.yaml', f'{BUNDLE}/recipe_v3.yaml')
for seed in [42, 137, 2026]:
    src = f'/content/logs/checkpoint/recipe_v3_seed{seed}'
    dst = f'{BUNDLE}/seed{seed}'
    os.makedirs(dst, exist_ok=True)
    # Copy only the final ckpt-30 to save space
    for f in glob.glob(f'{src}/ckpt-30.*') + [f'{src}/checkpoint']:
        if os.path.exists(f):
            shutil.copy(f, dst)
manifest = {
    'seeds': [42, 137, 2026],
    'recipe_v3': {'max_epoch': 30, 'tr_batch_sz': 320, 'tr_n_anchor': 160,
                  'f_min': 160.0, 'tr_seg_mode': 'random_oneshot', 'loss': 'NTxent'},
    'training_minutes_per_seed': results,
    'total_minutes': round(sum(results.values()), 1),
}
with open(f'{BUNDLE}/manifest.json', 'w') as f: json.dump(manifest, f, indent=2)
with tarfile.open('/content/recipe_v3_3seeds_output.tar.gz', 'w:gz') as tar:
    tar.add(BUNDLE, arcname='recipe_v3_3seeds_output')
sz = os.path.getsize('/content/recipe_v3_3seeds_output.tar.gz') / 1e6
print(f'\nBundle: /content/recipe_v3_3seeds_output.tar.gz ({sz:.1f} MB)')
print(json.dumps(manifest, indent=2))
print('\n=== ALL 3 SEEDS DONE ===')
print('Download via: Files panel (left sidebar) → recipe_v3_3seeds_output.tar.gz → right-click → Download')

## 12. (Optional) Auto-trigger browser download

In [ ]:
from google.colab import files
files.download('/content/recipe_v3_3seeds_output.tar.gz')